# multi_strat_009

> **Auto-generated documentation header.** Fill the `TODO` fields after review.
> Source notebook: `Notebook1 (156).ipynb`

| Field | Value |
|---|---|
| Section | `strategies` |
| Market | `unknown` |
| Trading style | `unknown` |
| Timeframe | `unknown` |
| Code cells | 13 |
| Detected functions | kalman_filter_fast, calculate_z_score, calculate_z_score |

### Research documentation
- **Hypothesis:** _TODO_
- **Alpha source:** _TODO_
- **Indicators / features:** _TODO_
- **Entry logic:** _TODO_
- **Exit logic (SL/TP):** _TODO_
- **Risk management:** _TODO_
- **Assumptions & limitations:** _TODO_
- **Performance (if available):** _TODO_

> Shared utilities live in `src/qresearch/` — prefer importing `jma_signals`, `backtest_trades`,
> `calculate_metrics`, `resample_ohlcv` instead of re-implementing them here.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
path = 'drive/My Drive/ipynb'


In [ ]:

# Use a raw string or replace backslashes with forward slashes
data1 = pd.read_csv(path+'/SBIN_minute_data_with_indicators.csv')  # Raw string
data2 = pd.read_csv(path+'/ICICIBANK_minute_data_with_indicators.csv')
# Display the DataFrame
data1
data2

In [ ]:
merged_data = pd.merge(data1, data2, left_on='date', right_on='date', how='inner')


In [ ]:
# Create a new DataFrame with Stock1, Stock2, and Date columns
data = pd.DataFrame({
    'Date': merged_data['date'],  # Include the date column
    'Stock1': merged_data['close_x'],  # Assuming 'close_x' represents Stock1 prices
    'Stock2': merged_data['close_y']   # Assuming 'close_y' represents Stock2 prices
})

# Set the Date column as the index (optional, but common for time series data)
data.set_index('Date', inplace=True)

# Display the resulting DataFrame
data


In [ ]:
# Reset the index to make 'Date' a regular column
data = data.reset_index()

# Verify the changes
print(data.head())


In [ ]:
!pip install pykalman vectorbt statsmodels


In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go

# Fast Kalman Filter Implementation
def kalman_filter_fast(price_data):
    n = len(price_data)
    filtered = np.zeros(n)

    # Initialize parameters
    state = price_data[0]  # Initial state
    covariance = 1.0  # Initial covariance
    process_var = 0.01  # Process variance (smoothness control)
    measurement_var = 3.0  # Measurement variance (noise)

    # Kalman Filter Iteration
    for t in range(n):
        # Prediction Step
        state_pred = state
        covariance_pred = covariance + process_var

        # Update Step
        kalman_gain = covariance_pred / (covariance_pred + measurement_var)
        state = state_pred + kalman_gain * (price_data[t] - state_pred)
        covariance = (1 - kalman_gain) * covariance_pred

        filtered[t] = state  # Save the filtered value

    return filtered

# Apply the fast Kalman Filter to Stock1 and Stock2
data['Stock1_Kalman'] = kalman_filter_fast(data['Stock1'].values)
data['Stock2_Kalman'] = kalman_filter_fast(data['Stock2'].values)

# Plot the results using Plotly
fig = go.Figure()

# Original Stock1
fig.add_trace(go.Scatter(
    x=data.index,
    y=data['Stock1'],
    mode='lines',
    name='Stock1 Original',
    line=dict(color='blue', width=1)
))

# Filtered Stock1
fig.add_trace(go.Scatter(
    x=data.index,
    y=data['Stock1_Kalman'],
    mode='lines',
    name='Stock1 Kalman Filter',
    line=dict(color='blue', dash='dot', width=2)
))

# Original Stock2
fig.add_trace(go.Scatter(
    x=data.index,
    y=data['Stock2'],
    mode='lines',
    name='Stock2 Original',
    line=dict(color='red', width=1)
))

# Filtered Stock2
fig.add_trace(go.Scatter(
    x=data.index,
    y=data['Stock2_Kalman'],
    mode='lines',
    name='Stock2 Kalman Filter',
    line=dict(color='red', dash='dot', width=2)
))

# Update layout for performance
fig.update_layout(
    title="Stock Prices with Optimized Kalman Filter",
    xaxis_title="Date",
    yaxis_title="Price",
    legend=dict(x=0, y=1, traceorder="normal"),
    template="plotly_dark",
    hovermode="x unified"
)

# Show the plot
fig.show()


In [ ]:
data

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go

# Optimized Z-score Calculation
def calculate_z_score(spread):
    rolling_mean = spread.rolling(window=60).mean()  # 30-period rolling mean
    rolling_std = spread.rolling(window=60).std()   # 30-period rolling std
    z_score = (spread - rolling_mean) / rolling_std
    return z_score

# Check for required columns
if 'Stock1_Kalman' not in data.columns or 'Stock2_Kalman' not in data.columns:
    raise ValueError("Columns 'Stock1_Kalman' and 'Stock2_Kalman' are missing in the dataset.")

# Compute Spread and Z-Score
data['Spread'] = data['Stock1_Kalman'] - data['Stock2_Kalman']
data['Z_Score'] = calculate_z_score(data['Spread'])

# Drop rows with NaN values resulting from rolling calculations
data = data.dropna()

# Plot 1: Z-Score with Threshold
fig_zscore = go.Figure()

# Z-Score Plot (using secondary y-axis)
fig_zscore.add_trace(go.Scatter(
    x=data.index,
    y=data['Z_Score'],
    mode='lines',
    name='Z-Score',
    line=dict(color='red', dash='dot', width=2)
))

# Add Threshold Lines for Z-Score (+2, -2)
fig_zscore.add_hline(y=2, line_dash="dot", line_color="green", annotation_text="Upper Threshold (+2)", annotation_position="top left")
fig_zscore.add_hline(y=-2, line_dash="dot", line_color="green", annotation_text="Lower Threshold (-2)", annotation_position="bottom left")

# Layout Configuration for Z-Score Plot
fig_zscore.update_layout(
    title="Z-Score with Thresholds",
    xaxis_title="Date",
    yaxis_title="Z-Score",
    template="plotly_dark",
    hovermode="x unified",
    height=600
)

# Display the Z-Score plot
fig_zscore.show()

# Plot 2: Spread
fig_spread = go.Figure()

# Spread Plot
fig_spread.add_trace(go.Scatter(
    x=data.index,
    y=data['Spread'],
    mode='lines',
    name='Spread',
    line=dict(color='blue', width=2)
))

# Layout Configuration for Spread Plot
fig_spread.update_layout(
    title="Spread: Stock1_Kalman - Stock2_Kalman",
    xaxis_title="Date",
    yaxis_title="Spread",
    template="plotly_dark",
    hovermode="x unified",
    height=600
)

# Display the Spread plot
fig_spread.show()


In [ ]:
import pandas as pd
import numpy as np

# Optimized Z-score Calculation (Vectorized)
def calculate_z_score(spread, window=60):
    rolling_mean = spread.rolling(window=window).mean()
    rolling_std = spread.rolling(window=window).std()
    return (spread - rolling_mean) / rolling_std

# Convert 'Date' column to datetime
data['Date'] = pd.to_datetime(data['Date'])

# Compute Spread and Z-Score in one go (vectorized)
data['Spread'] = data['Stock1_Kalman'] - data['Stock2_Kalman']
data['Z_Score'] = calculate_z_score(data['Spread'])

# Drop rows with NaN values resulting from rolling calculations
data.dropna(inplace=True)

# Initialize parameters
capital_per_trade = 100000  # ₹1,00,000 per trade
brokerage_fee = 0.001       # 0.1% brokerage
open_position = None        # No position initially
tradebook = []              # Tradebook to store trades
cumulative_pnl = 0          # Track cumulative PnL
capital = capital_per_trade # Initialize capital

# Pre-calculate entry prices for efficiency
data['Entry Price Stock1'] = data['Stock1_Kalman']
data['Entry Price Stock2'] = data['Stock2_Kalman']

# Vectorized backtest loop: Trade logic with vectorization
long_condition = data['Z_Score'] < -2
short_condition = data['Z_Score'] > 2
close_long_condition = data['Z_Score'] > 2
close_short_condition = data['Z_Score'] < -2

# Create signal columns for easy checking in the loop
data['long_signal'] = long_condition
data['short_signal'] = short_condition
data['close_long_signal'] = close_long_condition
data['close_short_signal'] = close_short_condition

# Loop over the data rows
for i in range(1, len(data)):
    # Extract Date and Time
    date = data['Date'].iloc[i]
    time = date.time()

    # Execute Long Position Entry
    if data['long_signal'].iloc[i] and open_position is None:
        entry_price_stock1 = data['Entry Price Stock1'].iloc[i]
        entry_price_stock2 = data['Entry Price Stock2'].iloc[i]
        position_size_stock1 = capital / entry_price_stock1  # Position size in Stock1
        position_size_stock2 = capital / entry_price_stock2  # Position size in Stock2
        tradebook.append({
            'Entry Date': date.date(),
            'Entry Time': time,
            'Entry Price Stock1': entry_price_stock1,
            'Entry Price Stock2': entry_price_stock2,
            'Position Size Stock1': position_size_stock1,
            'Position Size Stock2': position_size_stock2,
            'Trade Type': 'Long',
            'Capital Allocated': capital
        })
        open_position = 'Long'

    # Execute Short Position Entry
    elif data['short_signal'].iloc[i] and open_position is None:
        entry_price_stock1 = data['Entry Price Stock1'].iloc[i]
        entry_price_stock2 = data['Entry Price Stock2'].iloc[i]
        position_size_stock1 = capital / entry_price_stock1  # Position size in Stock1
        position_size_stock2 = capital / entry_price_stock2  # Position size in Stock2
        tradebook.append({
            'Entry Date': date.date(),
            'Entry Time': time,
            'Entry Price Stock1': entry_price_stock1,
            'Entry Price Stock2': entry_price_stock2,
            'Position Size Stock1': -position_size_stock1,  # Negative for short
            'Position Size Stock2': -position_size_stock2,  # Negative for short
            'Trade Type': 'Short',
            'Capital Allocated': capital
        })
        open_position = 'Short'

    # Close Long Position when Z-Score > 2
    if open_position == 'Long' and data['close_long_signal'].iloc[i]:
        exit_price_stock1 = data['Entry Price Stock1'].iloc[i]
        exit_price_stock2 = data['Entry Price Stock2'].iloc[i]
        pnl_stock1 = (exit_price_stock1 - entry_price_stock1) * position_size_stock1
        pnl_stock2 = (exit_price_stock2 - entry_price_stock2) * position_size_stock2
        total_pnl = pnl_stock1 + pnl_stock2
        total_brokerage = (entry_price_stock1 * position_size_stock1 + entry_price_stock2 * position_size_stock2) * brokerage_fee

        # Add details to tradebook
        tradebook[-1].update({
            'Exit Date': date.date(),
            'Exit Time': time,
            'Exit Price Stock1': exit_price_stock1,
            'Exit Price Stock2': exit_price_stock2,
            'PnL': total_pnl - total_brokerage,
            'Transaction Cost': total_brokerage,
            'Portfolio Value': capital + total_pnl - total_brokerage
        })
        cumulative_pnl += total_pnl - total_brokerage
        open_position = None  # Reset position after exit

    # Close Short Position when Z-Score < -2
    if open_position == 'Short' and data['close_short_signal'].iloc[i]:
        exit_price_stock1 = data['Entry Price Stock1'].iloc[i]
        exit_price_stock2 = data['Entry Price Stock2'].iloc[i]
        pnl_stock1 = (entry_price_stock1 - exit_price_stock1) * -position_size_stock1
        pnl_stock2 = (entry_price_stock2 - exit_price_stock2) * -position_size_stock2
        total_pnl = pnl_stock1 + pnl_stock2
        total_brokerage = (entry_price_stock1 * -position_size_stock1 + entry_price_stock2 * -position_size_stock2) * brokerage_fee

        # Add details to tradebook
        tradebook[-1].update({
            'Exit Date': date.date(),
            'Exit Time': time,
            'Exit Price Stock1': exit_price_stock1,
            'Exit Price Stock2': exit_price_stock2,
            'PnL': total_pnl - total_brokerage,
            'Transaction Cost': total_brokerage,
            'Portfolio Value': capital + total_pnl - total_brokerage
        })
        cumulative_pnl += total_pnl - total_brokerage
        open_position = None  # Reset position after exit

    # Close position at the end of the day at 15:28 if open
    if open_position is not None and time == pd.to_datetime("15:28:00").time():
        if open_position == 'Long':
            exit_price_stock1 = data['Entry Price Stock1'].iloc[i]
            exit_price_stock2 = data['Entry Price Stock2'].iloc[i]
            pnl_stock1 = (exit_price_stock1 - entry_price_stock1) * position_size_stock1
            pnl_stock2 = (exit_price_stock2 - entry_price_stock2) * position_size_stock2
            total_pnl = pnl_stock1 + pnl_stock2
            total_brokerage = (entry_price_stock1 * position_size_stock1 + entry_price_stock2 * position_size_stock2) * brokerage_fee
            tradebook[-1].update({
                'Exit Date': date.date(),
                'Exit Time': time,
                'Exit Price Stock1': exit_price_stock1,
                'Exit Price Stock2': exit_price_stock2,
                'PnL': total_pnl - total_brokerage,
                'Transaction Cost': total_brokerage,
                'Portfolio Value': capital + total_pnl - total_brokerage
            })
            cumulative_pnl += total_pnl - total_brokerage
            open_position = None

        elif open_position == 'Short':
            exit_price_stock1 = data['Entry Price Stock1'].iloc[i]
            exit_price_stock2 = data['Entry Price Stock2'].iloc[i]
            pnl_stock1 = (entry_price_stock1 - exit_price_stock1) * -position_size_stock1
            pnl_stock2 = (entry_price_stock2 - exit_price_stock2) * -position_size_stock2
            total_pnl = pnl_stock1 + pnl_stock2
            total_brokerage = (entry_price_stock1 * -position_size_stock1 + entry_price_stock2 * -position_size_stock2) * brokerage_fee
            tradebook[-1].update({
                'Exit Date': date.date(),
                'Exit Time': time,
                'Exit Price Stock1': exit_price_stock1,
                'Exit Price Stock2': exit_price_stock2,
                'PnL': total_pnl - total_brokerage,
                'Transaction Cost': total_brokerage,
                'Portfolio Value': capital + total_pnl - total_brokerage
            })
            cumulative_pnl += total_pnl - total_brokerage
            open_position = None

# Create DataFrame from tradebook
tradebook_df = pd.DataFrame(tradebook)

# Display the final tradebook
tradebook_df


In [ ]:
s = tradebook_df.to_csv('tradebook_df', index=False)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure the 'Entry Date', 'Exit Date', 'Entry Time', and 'Exit Time' are in datetime format
tradebook_df['Entry Date'] = pd.to_datetime(tradebook_df['Entry Date'])
tradebook_df['Exit Date'] = pd.to_datetime(tradebook_df['Exit Date'])
tradebook_df['Entry Time'] = pd.to_datetime(tradebook_df['Entry Time'], format='%H:%M:%S').dt.time
tradebook_df['Exit Time'] = pd.to_datetime(tradebook_df['Exit Time'], format='%H:%M:%S').dt.time

# Convert Entry Time and Exit Time to numerical values (in seconds since midnight)
tradebook_df['Entry Time in Seconds'] = pd.to_datetime(tradebook_df['Entry Time'].astype(str)).dt.hour * 3600 + pd.to_datetime(tradebook_df['Entry Time'].astype(str)).dt.minute * 60 + pd.to_datetime(tradebook_df['Entry Time'].astype(str)).dt.second
tradebook_df['Exit Time in Seconds'] = pd.to_datetime(tradebook_df['Exit Time'].astype(str)).dt.hour * 3600 + pd.to_datetime(tradebook_df['Exit Time'].astype(str)).dt.minute * 60 + pd.to_datetime(tradebook_df['Exit Time'].astype(str)).dt.second

# Calculate Cumulative PnL (Including PnL and Transaction Costs)
tradebook_df['Cumulative PnL'] = tradebook_df['PnL'].cumsum()

# Calculate Drawdown (from the Cumulative PnL)
tradebook_df['Max Drawdown'] = tradebook_df['Cumulative PnL'].cummax() - tradebook_df['Cumulative PnL']
tradebook_df['Drawdown'] = tradebook_df['Max Drawdown']

# Plot 1: Cumulative PnL with Drawdown Highlighted
plt.figure(figsize=(12, 6))
plt.plot(tradebook_df['Entry Date'], tradebook_df['Cumulative PnL'], label='Cumulative PnL', color='green')
plt.fill_between(tradebook_df['Entry Date'], tradebook_df['Cumulative PnL'],
                 tradebook_df['Cumulative PnL'] - tradebook_df['Drawdown'], color='red', alpha=0.3, label='Drawdown')
plt.title('Cumulative PnL with Drawdown')
plt.xlabel('Date')
plt.ylabel('Cumulative PnL')
plt.legend()
plt.show()

# Plot 2: PnL Distribution (Histogram with Fixed Bins)
plt.figure(figsize=(12, 6))
sns.histplot(tradebook_df['PnL'], bins=20, color='purple', kde=False)  # Use fixed bins, without KDE
plt.title('PnL Distribution')
plt.xlabel('PnL')
plt.ylabel('Frequency')
plt.show()

# Plot 3: Cumulative PnL over Time
plt.figure(figsize=(12, 6))
plt.plot(tradebook_df['Entry Date'], tradebook_df['Cumulative PnL'], color='blue')
plt.title('Cumulative PnL Over Time')
plt.xlabel('Date')
plt.ylabel('Cumulative PnL')
plt.show()

# Plot 4: Scatter Plot of Entry Time and Exit Time (only times in HH:MM:SS format)
plt.figure(figsize=(12, 6))
plt.scatter(tradebook_df['Entry Time in Seconds'], tradebook_df['Exit Time in Seconds'],
            c=tradebook_df['PnL'], cmap='coolwarm', alpha=0.7)
plt.title('Scatter Plot of Entry and Exit Time')
plt.xlabel('Entry Time (in seconds since midnight)')
plt.ylabel('Exit Time (in seconds since midnight)')
plt.colorbar(label='PnL')
plt.show()

# Plot 5: Drawdown Over Time
plt.figure(figsize=(12, 6))
plt.plot(tradebook_df['Entry Date'], tradebook_df['Drawdown'], color='red')
plt.title('Drawdown Over Time')
plt.xlabel('Date')
plt.ylabel('Drawdown')
plt.show()

# Plot 6: Histogram of Profit vs Exit Time (Avoid KDE here)
plt.figure(figsize=(12, 6))
sns.histplot(data=tradebook_df, x='Exit Time', weights='PnL', bins=20, color='blue', kde=False)  # Use fixed bins, without KDE
plt.title('Profit Distribution vs Exit Time')
plt.xlabel('Exit Time (HH:MM:SS)')
plt.ylabel('Profit (PnL)')
plt.show()
